# Exercises

We have prepared five exercises in this chapter:

1. Modify the HCM code to work for three groups. This exercise can be divded into four tasks: 
    - modify the parameters,
    - modify the calculate_u function,
    - execute the clustering,
    - plot the results.
2. For density clustering, plot the feature space with all element marked with different color, depending on the cluster that it's assigned to. You should do the following tasks:
    - fill the get_color method,
    - fill the plot code.
3. Build a method that plot baed on dendrograms_history and pydot, a dendrogram for the divisive clustering method. You should base on agglomerative method, but keep in mind that it works top-down instead of bottom-up. This exercise need just one function to be implemented:
    - show_tree_divisive. 
    You should loop over the dendrogram_history variable and loop over childs.
4. Implement the $s_{2}$ metric   
5. Draw the borders between clusters in the output image (for 5.0 grade)

*Author: Dawid Kryński*

## Libraries

To solve the exercises, we need the following libraries to load in the first place.


In [ ]:
import numpy
import random
import numpy as np
import pandas as pd
from math import sqrt

import matplotlib.image as img
from PIL import Image

from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from IPython.display import Image

import pickle

## Exercise 1: Modify the HCM code to work for three groups

The obvious part is the variable ```groups```, but the most changes needs to be done here:

In [ ]:
#%store -r data_set
with open("clustering/data_set.pkl", "rb") as f:
    data_set = pickle.load(f)

### change here:
groups = 3

error_margin = 0.01
m=2
assignation=np.zeros((len(data_set),groups))

centers = np.array([[0.01229673, 0.25183492],
       [0.3689626 , 0.61904127],
       [0.95732769, 0.45059586]])

centers = np.array([[0.01229673, 0.25183492],
       [0.3689626 , 0.61904127],
       [0.95732769, 0.45059586]])

def calculate_distance(x,v):
    return sqrt((x[0]-v[0])**2+(x[1]-v[1])**2)

def calculate_new_centers(u):
    new_centers=[]
    for c in range(groups):
        u_x_vector=np.zeros(2)
        u_scalar=0.0
        for i in range(len(data_set)):
            u_scalar = u_scalar+(u[i][c]**m)
            u_x_vector=np.add(u_x_vector,np.multiply(u[i][c]**m,data_set[i]))
        new_centers.append(np.divide(u_x_vector,u_scalar))
    return new_centers

def calculate_differences(new_assignation, assignation):     
    return np.sum(np.abs(np.subtract(assignation,new_assignation)))

def cluster_hcm(assignation,centers):
    difference_limit_not_achieved=True
    new_centers = centers
    iter=0
    while difference_limit_not_achieved:
        new_assignation=[]
        for i in range(len(data_set)):
            new_assignation.append(calculate_u_three(data_set[i]))
        new_centers = calculate_new_centers(new_assignation)
        if iter>0:
            if calculate_differences(new_assignation, assignation) < error_margin:
                difference_limit_not_achieved=False
        assignation=new_assignation
        iter=iter+1
    return new_assignation, new_centers

### Modify the ``calculate_u`` function

Fill the gap below to make the function working for more groups than two. The goal here is to calculate the distance between ``x`` and the center of a given group and append the value to ``minimal_distance``.

In [ ]:
def calculate_u_three(x):
    u_array = np.zeros(groups)
    # grab distance from x to each of the centres in one go
    minimal_distance = [calculate_distance(x, centers[g]) for g in range(groups)]
    min_group_id = np.argmin(minimal_distance)
    u_array[min_group_id] = 1
    return u_array

### Execute the clustering

As in the previous example we need to cluster it.

In [ ]:
new_assignation_hcm3, new_centers_hcm3 = cluster_hcm(assignation, centers)
pd.DataFrame(new_centers_hcm3)

### Plot the results

In [ ]:
red = data_set[np.where(np.array(new_assignation_hcm3)[:,0]==1)]
blue = data_set[np.where(np.array(new_assignation_hcm3)[:,1]==1)]
green = data_set[np.where(np.array(new_assignation_hcm3)[:,2]==1)]

fig, ax = plt.subplots()

ax.scatter(blue[:,0],blue[:,1],c='blue')
ax.scatter(red[:,0],red[:,1],c='red')
ax.scatter(green[:,0],green[:,1],c='green')
ax.scatter(np.array(new_centers_hcm3)[:,0],np.array(new_centers_hcm3)[:,1],c='black')
ax.set(xlabel='Seats count', ylabel='Distance range (km)',
       title='Aircrafts (clusters)')
ax.grid()
plt.show()

## Exercise 2: Plot the density clusters

Use the code below to plot the results. You can play with the max_distance variable to get more or less groups.

In [ ]:
#%store -r new_assignation_density
#%store -r data_set

with open("clustering/data_set.pkl", "rb") as f:
    data_set = pickle.load(f)

with open("clustering/assignation.pkl", "rb") as f:
    new_assignation_density = pickle.load(f)

### Fill the ``get_group_objects`` method

Only one line needs to be updated. The ``get_group_objects`` function should return the objects of a given group.

In [ ]:
colors = ['red','blue','green','orange','black','yellow']
    
def get_group_objects(color_id):
    # build a boolean mask and slice the dataset with it
    labels = np.argmax(np.asarray(new_assignation_density), axis=1)
    return data_set[labels == color_id]

### Fill the plot code

If done properly the code below should return a plot of two clusters and the noise.

In [ ]:
colors = ['red','blue','green','orange','black','yellow']

fig, ax = plt.subplots()

assigned_groups = new_assignation_density
for group in np.unique(assigned_groups):
    small_set = get_group_objects(group) 
    ax.scatter(small_set[:,0],small_set[:,1],c=colors.pop(0))
    for circle in small_set:
        circle1 = plt.Circle((circle[0],circle[1]), 0.25, color='green', fill=False)        
        ax.add_artist(circle1)
    
ax.set(xlabel='Seats count', ylabel='Distance range (km)',title='Aircrafts (clusters)')
ax.grid()
plt.show()

## Exercise 3: Implement the Dunn index:


In [ ]:
def dunn_index(assignation, data_set):
    points = np.asarray(data_set)
    # convert one-hot rows into plain cluster numbers if needed
    if np.ndim(assignation) > 1:
        cluster_of = np.argmax(assignation, axis=1)
    else:
        cluster_of = np.asarray(assignation)
    unique_clusters = np.unique(cluster_of)

    # bucket points by their cluster label
    buckets = {}
    for lbl in unique_clusters:
        buckets[int(lbl)] = points[cluster_of == lbl]

    # running max over all within-cluster pair distances
    diameter = 0.0
    for lbl in unique_clusters:
        pts = buckets[int(lbl)]
        for a in range(len(pts)):
            for b in range(a + 1, len(pts)):
                d = calculate_distance(pts[a], pts[b])
                if d > diameter:
                    diameter = d

    # running min over all cross-cluster pair distances
    separation = float('inf')
    for i, g1 in enumerate(unique_clusters):
        for g2 in unique_clusters[i + 1:]:
            for p in buckets[int(g1)]:
                for q in buckets[int(g2)]:
                    d = calculate_distance(p, q)
                    if d < separation:
                        separation = d

    return separation / diameter

Using data we used:

In [ ]:
import pickle

with open("clustering/data_set.pkl", "rb") as f:
    data_set = pickle.load(f)

new_assignation_hcm3 = np.array([[0., 1., 0.],[0., 1., 0.], [0., 1., 0.],[0., 1., 0.], [0., 1., 0.], [0., 1., 0.], [0., 0., 1.], [0., 0., 1.], [0., 0., 1.], [1., 0., 0.]])

calculate Dunn index:

In [ ]:
dunn_index(new_assignation_hcm3, data_set)